# skills mirror updater

Converts the opencode `SKILL.md` files into the static espanso trigger
mirror at `match/skills.yml`. Each `:<name>` trigger expands the **full
skill body**. Run top-to-bottom after adding or removing skills (the daily
cron does this automatically).

In [ ]:
import importlib
import tempfile
from pathlib import Path

import yaml
import skills_lib
importlib.reload(skills_lib)

skills = skills_lib.collect_skills()
print(f"Found {len(skills)} skills under {skills_lib.DEFAULT_SKILLS_DIR}")
print("output target:", skills_lib.DEFAULT_OUTPUT)
skills[:3]

In [ ]:
# Sanity checks: unique names, every skill has a non-empty body
assert len(skills) > 0, "no skills found"
missing = [s["id"] for s in skills if not s["name"] or not s["body"].strip()]
assert not missing, f"skills missing name/body: {missing}"
names = [s["name"] for s in skills]
assert len(names) == len(set(names)), "duplicate skill names would collide"
print("All skills unique and have a non-empty body")

In [ ]:
# Generate to a temp file and validate each trigger == the full body
with tempfile.TemporaryDirectory() as d:
    tmp = Path(d) / "skills.yml"
    skills_lib.generate_skills_yml(skills, output_path=tmp)
    parsed = yaml.safe_load(tmp.read_text())
    triggers = {m["trigger"]: m["replace"] for m in parsed["matches"]}
assert ":skills" in triggers
assert len(parsed["matches"]) == len(skills) + 1  # + the :skills listing
for s in skills:
    assert triggers[":" + s["name"]] == s["body"], f"body mismatch for {s['name']}"
print(f"All {len(skills)} triggers expand the full body")
tdd = next(s for s in skills if s["name"] == "tdd")
print(f":tdd -> {len(tdd['body'])} chars / {tdd['body'].count(chr(10))+1} lines")

In [ ]:
# Preview the :skills listing
print(skills_lib.format_skill_list(skills))

In [ ]:
# Regenerate the real mirror at match/skills.yml
out = skills_lib.generate_skills_yml(skills)
print(f"Wrote {out}")
print(out.read_text()[:600])

In [ ]:
# End-to-end: run the same CLI command the cron runs
import subprocess, sys

r = subprocess.run([sys.executable, "skills_lib.py", "sync"], capture_output=True, text=True)
assert r.returncode == 0, r.stderr
print(r.stdout)
print("CLI e2e OK")